In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 22:41:25.634100: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 22:41:26.545403: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'],
        "ports": [50151, 50152, 50153]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 22:41:31,638 [DEBUG] [Rain] Rain is initialized
2023-07-04 22:41:31,640 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 22:41:31,641 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-04 22:41:31,642 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 22:41:31,643 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 22:41:31,644 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-04 22:41:31,645 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-04 22:41:31,646 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 22:41:31,653 [DEBUG] [Rain] Creating workers
2023-07-04 22:41:31,659 [INFO] [Provisioner] provisioner is serving
2023-07-04 22:41:31,660 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 22:41:31,662 [INFO] [Coordinator] coordinator is serving
2023-07-04 22:41:31,663 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 22:41:31,667 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 22:41:31,668 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 22:41:31,670 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 22:41:31,672 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-04 22:41:31,674 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 22:41:31,675 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-04 22:41:31,678 [INFO] [Worker_50

157/157 [==============================] - 1s 4ms/step - loss: 0.7066 - accuracy: 0.7778


2023-07-04 22:41:45,589 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:41:45,590 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider


2023-07-04 22:41:45,677 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 22:41:45,685 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
2023-07-04 22:41:45,706 [DEBUG] [DeepLearning] Iteration 1/3 complete for worker 3.
2023-07-04 22:41:45,707 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-04 22:41:45,708 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07-04 22:41:45,709 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration2 to worker3
2023-07-04 22:41:45,709 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/3.pkl to worker3
2023-07-04 22:41:45,791 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-04 22:41:45,791 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker3
2023-07-04 22:41:45,792 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3


...

2023-07-04 22:41:47,013 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-04 22:41:47,104 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 22:41:47,111 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
DEBUG:DeepLearning:Asynchronous update is done by worker 3
2023-07-04 22:41:47,133 [DEBUG] [DeepLearning] Iteration 2/3 complete for worker 3.
DEBUG:DeepLearning:Iteration 2/3 complete for worker 3.
2023-07-04 22:41:47,134 [DEBUG] [DeepLearning] Starting iteration 3/3
DEBUG:DeepLearning:Starting iteration 3/3
2023-07-04 22:41:47,135 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
DEBUG:DividerAmbassador:127.0.0.1:50153
2023-07-04 22:41

157/157 [==============================] - 1s 3ms/step - loss: 0.2628 - accuracy: 0.9201


2023-07-04 22:41:56,802 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:41:56,803 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to divider


2023-07-04 22:41:56,919 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-04 22:41:56,928 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
DEBUG:DeepLearning:Asynchronous update is done by worker 2
2023-07-04 22:41:56,950 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 2.
2023-07-04 22:41:57,530 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:41:57,531 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-04

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1468 - accuracy: 0.9556

Test accuracy: 95.6%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 22:41:59,101 [DEBUG] [Rain] Creating workers
DEBUG:Rain:Creating workers
2023-07-04 22:41:59,104 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-04 22:41:59,105 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-04 22:41:59,107 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-04 22:41:59,108 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-04 22:41:59,110 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 22:41:59,111 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-04 22:41:59,

157/157 [==============================] - 2s 5ms/step - loss: 0.2113 - accuracy: 0.9359


2023-07-04 22:42:15,960 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:42:15,962 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:42:15,963 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:42:15,964 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3


sending data to divider
sending data to divider


2023-07-04 22:42:16,137 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_1_trained.pkl from worker3 successfully
2023-07-04 22:42:16,137 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_1_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_1_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_1_trained.pkl from worker2 successfully
2023-07-04 22:42:16,161 [DEBUG] [DeepLearning] Iteration 1/3 complete.
DEBUG:DeepLearning:Iteration 1/3 complete.
2023-07-04 22:42:16,162 [DEBUG] [DeepLearning] Starting iteration 2/3
DEBUG:DeepLearning:Starting iteration 2/3
2023-07-04 22:42:16,182 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-04 22:42:16,183 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-04 22:42:16,183 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
DEBUG:DividerAmbassador:127.0.0.1:50151
2023-07-04 22:42:16,185 [DEBUG] [DividerAmbassador] divide

...

2023-07-04 22:42:21,485 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:42:21,489 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2


sending data to divider
157/157 [==============================] - 2s 7ms/step - loss: 0.1779 - accuracy: 0.9454


2023-07-04 22:42:21,562 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:42:21,564 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-04 22:42:21,574 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}


sending data to divider
sending data to divider


DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 22:42:21,576 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
2023-07-04 22:42:21,784 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-04 22:42:21,823 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 22:42:21,824 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/d

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1102 - accuracy: 0.9657

Test accuracy: 96.6%


2023-07-04 22:44:23,120 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
2023-07-04 22:44:23,120 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
INFO:Worker_50152:Running the worker with id: 2 on iteration: 1


sending data to divider


2023-07-04 22:44:44,873 [DEBUG] [Coordinator] coordinator is sending workers info to divider
DEBUG:Coordinator:coordinator is sending workers info to divider
2023-07-04 22:44:44,912 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 1
2023-07-04 22:44:44,913 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
2023-07-04 22:44:44,912 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 1
2023-07-04 22:44:44,913 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
INFO:Worker_50153:Running the worker with id: 3 on iteration: 1
INFO:Worker_50152:Running the worker with id: 2 on iteration: 1


sending data to divider
sending data to divider


...

sending data to divider
sending data to divider
sending data to divider
sending data to divider
sending data to divider
sending data to divider


2023-07-04 22:46:15,654 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 18
2023-07-04 22:46:15,654 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 18
INFO:Worker_50151:Running the worker with id: 1 on iteration: 18
2023-07-04 22:46:15,661 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 18
2023-07-04 22:46:15,661 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 18
INFO:Worker_50153:Running the worker with id: 3 on iteration: 18
2023-07-04 22:46:15,765 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 19
2023-07-04 22:46:15,765 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 19
INFO:Worker_50151:Running the worker with id: 1 on iteration: 19
2023-07-04 22:46:15,776 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 19
2023-07-04 22:46:15,776 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 19
INFO:Worker_50153:Running the worker with id: 3 on iteration:

sending data to divider
sending data to divider
sending data to divider
sending data to divider
sending data to divider


2023-07-04 22:46:34,680 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-04 22:46:34,680 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-04 22:46:34,682 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
INFO:Worker_50151:Running the worker with id: 1 on iteration: 1
2023-07-04 22:46:34,682 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
INFO:Worker_50152:Running the worker with id: 2 on iteration: 1


sending data to divider
sending data to divider
